<div dir="rtl">

# אימון מחדש של גלאי האבטחה (`security.pt`): סכין, אקדח, אש, עשן

**למה:** במדידה על UCF-Crime ‏(`live/results/ucf_2026-11-24`) הגלאי הנוכחי לא ראה סכין באף אחד מ-46 האירועים, וראה אקדח רק ב-20% מהם.

**מה המחברת עושה, בלי שום תיוג ידני:**
1. מורידה שלושה דאטהסטים ציבוריים:
   - אקדחים ממצלמות אבטחה אמיתיות.
   - **WeaponDetection** מ-Hugging Face: כ-9,600 תמונות עם הרבה סכינים ואקדחים.
   - D-Fire לאש ולעשן, כדי שהגלאי לא ישכח אותם.
2. לכל תמונה של סכין או אקדח מוסיפה עותק "מצלמת אבטחה": רזולוציה נמוכה, טשטוש, דחיסה חזקה, ולפעמים שחור-לבן. ככה נראים הפריימים ב-UCF.
3. ממשיכה לאמן מ-`security.pt` הנוכחי, כך שמה שהוא כבר יודע (אש ועשן) נשמר. האימון נשמר ב-Drive, ואם Colab מתנתק מריצים שוב והוא ממשיך מאיפה שעצר.
4. **הבדיקה האמיתית:** מריצה את הגלאי החדש על סרטוני UCF ששמורים ב-Drive ‏(`live_eval/ucf_videos`), ומשווה לגלאי הישן עם האבחון ומדידת ההתרעות. UCF לא נכנס לאימון, כך שזו בדיקה הוגנת.

**לפני שמתחילים:** Runtime → Change runtime type → **T4 GPU**. ואז Runtime → Run all.

**זמן:** הורדה ובנייה כ-20 דקות, אימון כ-1.5-2 שעות, ובדיקה על UCF כמו תא 5 במחברת המדידה. מומלץ טוקן של Hugging Face ב-Secrets ‏(`HF_TOKEN`, עם Notebook access דלוק), אחרת ההורדה עלולה להיחסם לזמן מה.

</div>

In [ ]:
# 1. GPU, Drive, packages
%cd /content
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ none: Runtime → Change runtime type → T4 GPU')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
RUNS = f'{DRIVE}/security_train'            # training checkpoints: survive a disconnect
os.makedirs(RUNS, exist_ok=True)
!pip -q install ultralytics lap datasets huggingface_hub pyarrow
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF token: ok')
except Exception:
    print('no HF_TOKEN secret: continuing without it')

In [ ]:
# 2. Code (the branch with the new dataset script)
BRANCH = 'live-action-alerts'
%cd /content
!rm -rf /content/final_project
!git clone -q -b {BRANCH} https://github.com/Talyaavraham/final_project.git /content/final_project
%cd /content/final_project/backend
import sys; sys.path.insert(0, 'train')
!ls security.pt train/security_dataset.py live/perceive.py

In [ ]:
# 3. Download the sources (~5 GB, a few minutes)
from huggingface_hub import hf_hub_download, snapshot_download
import datasets
hf_hub_download('jsalazar/US-Real-time-gun-detection-in-CCTV-An-open-problem-dataset', 'weapons_images_2fps.zip',
                repo_type='dataset', local_dir='/content/src/weapons')
snapshot_download('badsaarow/d-fire', repo_type='dataset', local_dir='/content/src/dfire', allow_patterns=['data/*.parquet'])
wd = datasets.load_dataset('Subh775/WeaponDetection')
print(wd)
print('objects:', wd['train'].features['objects'])

In [ ]:
# 4. Build one dataset: gun, knife, fire, smoke
import importlib, security_dataset as sd
importlib.reload(sd)
sd.WEAPONS_ZIP = '/content/src/weapons/weapons_images_2fps.zip'
sd.DFIRE_DIR = '/content/src/dfire/data'
!rm -rf /content/security/dataset
w = sd.Writer('/content/security/dataset')
sd.add_weapons(w)
sd.add_dfire(w, limit=8000)          # enough to keep fire / smoke; the knife and gun data are the point
base_val = list(w.files['val'])      # CCTV guns + D-Fire: the images the old security.pt was validated on
sd.add_weapondet(w, wd, cctv=True)
for z in sorted(__import__('glob').glob(f'{DRIVE}/security_exports/*.zip')):   # optional: knives.zip etc.
    import zipfile; root = '/content/own_' + os.path.basename(z)[:-4]
    zipfile.ZipFile(z).extractall(root); sd.add_own(w, root)
w.yaml(); w.yaml('data_base.yaml', base_val)
print('images:', w.n, '| boxes:', w.boxes)

In [ ]:
# 5. Train: continue from the current security.pt. Resumes from Drive after a disconnect.
#    ~4 min/epoch on a T4. In a hurry: EPOCHS = 10.
from ultralytics import YOLO
EPOCHS = 20
last = f'{RUNS}/security_v2/weights/last.pt'
if os.path.exists(last):
    print('resuming', last)
    YOLO(last).train(resume=True)
else:
    YOLO('security.pt').train(data='/content/security/dataset/data.yaml', epochs=EPOCHS, imgsz=800, batch=16,
                              device=0, project=RUNS, name='security_v2', exist_ok=True, seed=0,
                              close_mosaic=4, scale=0.7, patience=6)
NEW = f'{RUNS}/security_v2/weights/best.pt'

In [ ]:
# 6. Per-class numbers on the validation images: new model vs the old security.pt
from ultralytics import YOLO
NEW = f'{RUNS}/security_v2/weights/best.pt'
for title, weights in (('OLD security.pt', 'security.pt'), ('NEW security_v2', NEW)):
    for vname, yaml in (('all validation', 'data.yaml'), ('base: CCTV guns + D-Fire', 'data_base.yaml')):
        r = YOLO(weights).val(data='/content/security/dataset/' + yaml, imgsz=800, plots=False, verbose=False)
        print(f'\n{title} | {vname}')
        for k, c in enumerate(r.ap_class_index):
            print(f'  {r.names[int(c)]:6s} P={r.box.p[k]:.3f} R={r.box.r[k]:.3f} mAP50={r.box.ap50[k]:.3f}')

<div dir="rtl">

## הבדיקה האמיתית: UCF-Crime

מריצים את המצלמה החיה עם הגלאי החדש על אותם סרטונים מ-`live_eval/ucf_videos`, ושומרים ל-`live_eval/perception_fps3_sec2`. המסקנה נקבעת לפי המספרים כאן, לא לפי תא 6.

</div>

In [ ]:
# 7. UCF-Crime with the new detector (same videos and fps as the first measurement; resumable)
import shutil
FPS = 3
LE = f'{DRIVE}/live_eval'
shutil.copy(NEW, 'security_v2.pt')
os.environ['LIVE_EXTRA'] = 'security_v2.pt'
os.environ['LIVE_DEVICE'] = '0'
!python live/perceive.py --gt eval/ground_truth_uca.json --videos {LE}/ucf_videos --out {LE}/perception_fps{FPS}_sec2 --fps {FPS}

In [ ]:
# 8. Old vs new detector: diagnosis + alerts
for tag, perc in (('OLD', f'{LE}/perception_fps{FPS}'), ('NEW', f'{LE}/perception_fps{FPS}_sec2')):
    print(f'\n#################### {tag} detector')
    !python live/diagnose.py --gt eval/ground_truth_uca.json --perception {perc} --name diagnose_{tag}
    !python live/eval_alerts.py --gt eval/ground_truth_uca.json --perception {perc} --scenarios knife,weapon,snatch --name alerts_{tag}
!mkdir -p {LE}/sec2 && cp live/results/diagnose_* live/results/alerts_* {LE}/sec2/

In [ ]:
# 9. Download the new model (save it as backend/security_v2.pt) and the results
from google.colab import files
files.download(NEW)
import glob
for p in sorted(glob.glob('live/results/diagnose_*.txt') + glob.glob('live/results/alerts_*.txt')):
    files.download(p)